<a href="https://colab.research.google.com/github/gbrixi/minerva/blob/main/examples/notebooks/loci_viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minerva loci viewer

Multimodal contact fingerprints on a genomic locus — **base pairing** (RNA, coral) · **repeat** (DNA, indigo) · **protein** (teal).

Two views: **heads** (fast, one forward pass) and **Jacobian fingerprint** (detailed, slower).
Ships **UG27** and **TwoAYGGAY** examples; you can also upload your own GenBank (`.gb`).

**Fill the form below, then `Runtime` → `Run all`.**

In [ ]:
#@title Setup — install &amp; import { display-mode: "form" }
import importlib.util, os, sys
IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/gbrixi/minerva.git"  #@param {type:"string"}
#@markdown flash-attn is **optional** — leave off to use torch SDPA (no install, works on CPU too; flash is a bit faster on GPU).
install_flash_attn = False  #@param {type:"boolean"}
if IN_COLAB and not os.path.isfile("MINERVA_READY"):
    !pip -q install biopython plotly
    if install_flash_attn and importlib.util.find_spec("flash_attn") is None:
        !pip -q install flash-attn --no-build-isolation
    if not os.path.isdir("minerva"):
        !git clone -q $REPO_URL
    open("MINERVA_READY", "w").close()
for _p in (".", "..", "minerva", os.path.expanduser("~/minerva")):
    if os.path.isdir(os.path.join(_p, "minerva")):
        sys.path.insert(0, os.path.abspath(_p)); break

# Drop any already-imported Minerva modules/objects so rerunning setup after a
# code update cannot keep stale code or an old fp16-loaded model in memory.
for _name in list(sys.modules):
    if _name == "minerva" or _name.startswith("minerva."):
        sys.modules.pop(_name, None)
for _old in ("model", "tokenizer", "lora_model", "trainer"):
    globals().pop(_old, None)
import time, torch, numpy as np, matplotlib.pyplot as plt
from minerva.visualization import (head_contacts_rgb, render_fingerprints,
                                   publication_head_contacts_rgb,
                                   plot_locus, plot_publication_locus,
                                   interactive_overlay)
from minerva.data import extract_and_tokenize_gb
import minerva.modeling_minerva as _mm

# Force the torch SDPA path when flash-attn is not explicitly requested.
# This avoids Colab/T4 using a preinstalled or half-installed flash-attn build.
if not globals().get("install_flash_attn", False):
    _mm._HAS_FLASH = False
    _mm._HAS_FLASH_ROTARY = False

import inspect
_rotary_forward_src = inspect.getsource(_mm.RotaryEmbedding.forward)
if 'q.device.type != "cuda"' not in _rotary_forward_src or 'return self.apply_torch(q, k)' not in _rotary_forward_src:
    raise RuntimeError("Imported Minerva lacks the non-flash rotary fallback. Restart and import patched code.")
_rmsnorm_src = inspect.getsource(_mm.rmsnorm_func)
if 'hidden_states.float()' not in _rmsnorm_src:
    raise RuntimeError("Imported Minerva has fp16-unsafe RMSNorm. Restart and import patched code.")
_rms_x = torch.full((1, 4), 322.0, dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
_rms_w = torch.ones(4, dtype=_rms_x.dtype)
_rms_y = _mm.rmsnorm_func(_rms_x, _rms_w, torch.tensor(1e-5))
if (not torch.isfinite(_rms_y).all()) or bool(torch.all(_rms_y == 0)):
    raise RuntimeError("Imported Minerva failed the fp16 RMSNorm runtime check. Restart runtime and rerun setup.")
print("ready | colab:", IN_COLAB, "| flash-attn enabled:", _mm._HAS_FLASH, "(SDPA fallback if False)")

In [ ]:
#@title Hugging Face login (private model) { display-mode: "form" }
#@markdown `gbrixi/minerva-1` is **private** — you need read access + an HF token.
#@markdown In Colab, add it as a Secret named `HF_TOKEN`; on a cluster, `hf auth login`
#@markdown or a cached Hugging Face token will be reused automatically.
from huggingface_hub import get_token, login
_tok = None
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except Exception:
    _tok = os.environ.get("HF_TOKEN")
_tok = _tok or get_token()
if _tok:
    login(token=_tok, new_session=False)
else:
    login(new_session=False)   # interactive prompt if no token is cached

In [ ]:
#@title 1. Choose locus &amp; view { display-mode: "form", run: "auto" }
#@markdown Pick an example (or **upload your own** in the next cell), the view, and the head set.
example  = "ug27"           #@param ["ug27", "twoayggay", "upload your own"]
view     = "heads (fast)"   #@param ["heads (fast)", "jacobian (detailed)", "both"]
head_set = "l2 (last-2)"    #@param ["l2 (last-2)", "l6 (last-6)"]
renderer = "publication (PDF)"   #@param ["publication (PDF)", "static (PDF)", "interactive (zoom/pan)"]
contrast = "raw probabilities"   #@param ["raw probabilities", "auto contrast"]
#@markdown Window in **token** positions — leave both `0` for the whole locus. `record` applies to your own upload.
record       = 0  #@param {type:"integer"}
window_start = 0  #@param {type:"integer"}
window_end   = 0  #@param {type:"integer"}

In [ ]:
#@title Advanced settings { display-mode: "form", run: "auto" }
MODEL = "gbrixi/minerva-1"  #@param {type:"string"}
jac_max_tokens = 384  #@param {type:"integer"}
dpi = 600             #@param {type:"integer"}
DEVICE, DTYPE = "cuda", torch.bfloat16

In [ ]:
#@title (optional) Upload your own GenBank { display-mode: "form" }
#@markdown Run this only if you chose **upload your own** above.
UPLOADED = None
if IN_COLAB and example == "upload your own":
    from google.colab import files
    up = files.upload()
    if up:
        UPLOADED = list(up.keys())[0]
        print("uploaded:", UPLOADED)

In [ ]:
# Load the Minerva model + tokenizer (once). No trust_remote_code needed:
# the tokenizer is a stock PreTrainedTokenizerFast, and the model comes from the
# installed `minerva` package (not remote code fetched from the Hub).
from transformers import AutoTokenizer
from minerva.modeling_minerva import MinervaForMaskedLM
if "model" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = MinervaForMaskedLM.from_pretrained(MODEL, torch_dtype=DTYPE).to(DEVICE).eval()
vocab = tokenizer.get_vocab()
nuc_ids  = [vocab[c] for c in "atgc" if c in vocab]
aa_chars = list("ACDEFGHIKLMNPQRSTVWY")
aa_ids   = [vocab[c] for c in aa_chars if c in vocab]
print(type(model).__name__, "| heads:", sorted(model.linear_heads.keys()))

In [ ]:
def _find(name):
    for base in ("data", "examples/data", "notebooks/data", "minerva/notebooks/data",
                 os.path.expanduser("~/minerva/examples/data")):
        if os.path.exists(os.path.join(base, name)):
            return os.path.join(base, name)
    raise FileNotFoundError(name)

PRESETS = {
    "ug27":      dict(gb=_find("UG27_systems.gb"), record=2, window=(748, 1772)),
    "twoayggay": dict(gb=_find("TwoAYGGAY_Pseudomonas_fluorescens_SBW25.gb"), record=0, window=None),
}
if example == "upload your own":
    assert UPLOADED, "Run the upload cell first (or pick a built-in example)."
    cfg = dict(gb=UPLOADED, record=record, window=None)
else:
    cfg = dict(PRESETS[example])
if window_end > window_start:            # form window overrides the preset when set
    cfg["window"] = (window_start, window_end)

record_obj = extract_and_tokenize_gb(cfg["gb"], use_existing_translations=True)[cfg["record"]]
sequence = record_obj["sequence"]
all_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(sequence))
ntok = len(all_tokens)
window = cfg["window"]
suffix = "_l6" if head_set.startswith("l6") else ""
locus = record_obj["locus_name"][:44]
print("locus :", locus)
print("tokens:", ntok, "| window:", window or "(whole)", "| heads:", head_set, "| render:", renderer)

def render(rgb, tokens, title, tag):
    if renderer.startswith("interactive"):
        interactive_overlay(rgb, title=title).show()
    else:
        plot_locus(rgb, tokens=tokens, title=title, save=f"{tag}.pdf", dpi=dpi)
        plt.show(); print("saved", f"{tag}.pdf")

In [ ]:
# --- Heads view (fast) ---
if view.startswith("heads") or view == "both":
    heads = [f"base_pairing{suffix}", f"repeat{suffix}", f"protein{suffix}"]
    kw = dict(sequence=sequence, tokenizer=tokenizer, head_names=heads)
    if window:
        kw.update(seed_start=window[0], seed_end=window[1])
    pr = model.predict_contacts(**kw)["predictions"]
    chan = {h.replace(suffix, ""): pr[h].float().cpu().numpy() for h in heads}
    toks = all_tokens[window[0]:window[1]] if window else all_tokens
    print("head ranges:", {k: (float(np.nanmin(v)), float(np.nanmax(v))) for k, v in chan.items()})
    if renderer.startswith("publication"):
        use_auto_contrast = contrast.startswith("auto")
        rgb_heads, contrast_stats = publication_head_contacts_rgb(
            chan, tokens=toks, auto_contrast=use_auto_contrast, return_stats=True)
        print("contrast:", contrast_stats if use_auto_contrast else "raw probabilities")
    else:
        rgb_heads = head_contacts_rgb(chan, tokens=toks)   # token-aware masking
    nonwhite = int(np.sum(np.any(rgb_heads < 0.995, axis=-1)))
    print("visible pixels:", nonwhite, "of", rgb_heads.shape[0] * rgb_heads.shape[1])
    if nonwhite == 0:
        print("warning: selected heads produced no visible structure; try l6 or jacobian mode")
    render(rgb_heads, toks, f"Heads {head_set} — {locus} {window or '(whole)'}", "heads", overlay_kind="heads")


In [ ]:
# --- Jacobian fingerprint view (detailed, slower) ---
if view.startswith("jacobian") or view == "both":
    if window:
        js, je = window[0], min(window[1], window[0] + jac_max_tokens)
    else:
        c = ntok // 2; js, je = max(0, c - jac_max_tokens // 2), min(ntok, c + jac_max_tokens // 2)
    print(f"jacobian window [{js}:{je}] = {je - js} tokens (this runs one forward per position/token)...")
    t0 = time.time()
    fp = model.get_fingerprints(
        sequence, tokenizer, nuc_token_ids=nuc_ids, aa_token_ids=aa_ids,
        max_batch_size=32, position_range=(js, je), show_progress=True,
        jac_aa_order=aa_chars)
    jac_tokens = fp.tokens
    style = "publication" if renderer.startswith("publication") else "default"
    rgb_jac = render_fingerprints(fp, style=style)
    print(f"done in {time.time() - t0:.0f}s")
    render(rgb_jac, jac_tokens, f"Jacobian fingerprint — {locus} [{js}:{je}]", "jacobian", overlay_kind="fingerprint")


In [ ]:
#@title Download figures { display-mode: "form" }
if IN_COLAB:
    from google.colab import files
    for f in ("heads.pdf", "jacobian.pdf"):
        if os.path.exists(f):
            files.download(f)

---
**Notes**
- Default output is a single-panel publication PDF using raw 0..1 contact probabilities, the Minerva RNA/PDB/repeat/Jacobian palette, and 600 dpi export.
- The general viewer does not add the UG27-specific dot plot, triptych, or inset panels.
- Dot plots are available from code via `minerva.visualization.plot_dotplot(...)` for users who want forward/reverse-complement repeat diagnostics.
- Auto contrast is available for exploratory viewing, but raw probabilities are the publication-equivalent default.
- Heads view is token-masked (protein only on amino-acid pairs; base-pairing/repeat only on nucleotide pairs), matching the Jacobian convention.
- **Static** saves a PDF with token-type tracks; **interactive** gives Plotly zoom/pan/hover for big maps.
- Heads view = one forward pass; Jacobian view is the slow, detailed one (window capped to `jac_max_tokens`).
- flash-attn is optional (torch SDPA fallback); `trust_remote_code` is not needed.

**Private access:** the model stays private. To let someone run this, grant their HF username **read access** on `gbrixi/minerva-1` (repo → Settings → collaborators, or via an HF org), have them make a read **token**, and add it as the Colab Secret `HF_TOKEN` (or paste it at the login prompt).
